In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

episodios = spark.table("mvp_engdados.bronze.episodes_raw")
imdb = spark.table("mvp_engdados.bronze.imdb_raw")

print("Episódios:", episodios.count())
print("IMDb:", imdb.count())

display(
    episodios.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

Episódios: 1510
IMDb: 1495


series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


In [0]:
def mostrar_nulos(df, nome_tabela):
    colunas_texto = [
        campo.name
        for campo in df.schema.fields
        if isinstance(campo.dataType, StringType)
    ]

    expressoes = [
        F.sum(
            F.when(
                F.col(coluna).isNull() | (F.trim(F.col(coluna)) == ""),
                1
            ).otherwise(0)
        ).alias(coluna)
        for coluna in colunas_texto
    ]

    print(f"Valores nulos ou vazios — {nome_tabela}")
    display(df.agg(*expressoes))

mostrar_nulos(episodios, "episodes_raw")
mostrar_nulos(imdb, "imdb_raw")

Valores nulos ou vazios — episodes_raw


season,episode_num_in_season,episode_num_overall,title,directed_by,written_by,original_air_date,prod_code,us_viewers,series_id,_source_file,featured_character
0,0,0,0,0,0,0,433,5,0,0,1389


Valores nulos ou vazios — imdb_raw


season,episode_num,title,original_air_date,imdb_rating,total_votes,desc,series_id,_source_file
0,0,0,0,0,0,0,0,0


In [0]:
display(
    episodios
    .groupBy("series_id", "season", "episode_num_in_season")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num_in_season")
)

display(
    imdb
    .groupBy("series_id", "season", "episode_num")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num")
)

series_id,season,episode_num_in_season,count
lost,6,12,2


series_id,season,episode_num,count


In [0]:
display(
    imdb.select("original_air_date").distinct().show(30, truncate=False)
)

+-----------------+
|original_air_date|
+-----------------+
|13 Oct. 1994     |
|4 May 1995       |
|11 May 1995      |
|18 Dec. 1997     |
|15 Jan. 1998     |
|16 Apr. 1998     |
|24 Sep. 1998     |
|11 Feb. 1999     |
|22 Apr. 1999     |
|30 Sep. 1999     |
|25 Nov. 1999     |
|10 May 2001      |
|17 Jan. 2002     |
|17 Oct. 2002     |
|16 Jan. 2003     |
|1 May 2003       |
|27 Oct. 1994     |
|2 Mar. 1995      |
|12 Oct. 1995     |
|1 Feb. 1996      |
|15 Feb. 1996     |
|26 Sep. 1996     |
|3 Oct. 1996      |
|13 Feb. 1997     |
|24 Apr. 1997     |
|30 Oct. 1997     |
|20 Nov. 1997     |
|17 Dec. 1998     |
|7 Oct. 1999      |
|16 May 2002      |
+-----------------+
only showing top 30 rows


In [0]:
display(
    imdb
    .withColumn("rating_num", F.col("imdb_rating").cast("double"))
    .withColumn("votes_num", F.col("total_votes").cast("long"))
    .groupBy("series_id")
    .agg(
        F.min("rating_num").alias("menor_nota"),
        F.max("rating_num").alias("maior_nota"),
        F.avg("rating_num").alias("nota_media"),
        F.min("votes_num").alias("menor_numero_votos"),
        F.max("votes_num").alias("maior_numero_votos")
    )
    .orderBy("series_id")
)

display(
    episodios
    .withColumn("viewers_num", F.col("us_viewers").cast("double"))
    .groupBy("series_id")
    .agg(
        F.min("viewers_num").alias("menor_audiencia"),
        F.max("viewers_num").alias("maior_audiencia"),
        F.avg("viewers_num").alias("audiencia_media")
    )
    .orderBy("series_id")
)

series_id,menor_nota,maior_nota,nota_media,menor_numero_votos,maior_numero_votos
big_bang_theory,6.7,9.5,7.813214285714288,1888,8791
breaking_bad,7.9,10.0,9.02741935483871,16545,150341
brooklyn_99,4.8,9.6,8.127450980392155,1178,6225
friends,7.1,9.7,8.422553191489362,3427,14033
game_of_thrones,4.1,9.9,8.753424657534246,25968,242548
how_i_met_your_mother,5.5,9.5,8.141346153846156,2678,18191
lost,7.2,9.7,8.547899159663869,843,12557
the_office,6.4,9.8,8.219148936170212,2724,15245
the_walking_dead,4.1,9.6,7.915819209039551,4325,42427


series_id,menor_audiencia,maior_audiencia,audiencia_media
big_bang_theory,7340000.0,2.044E7,1.4646308243727598E7
breaking_bad,970000.0,1.028E7,2324385.964912281
brooklyn_99,1310000.0,1.507E7,2620718.954248366
friends,1.565E7,5.29E7,2.539080508474576E7
game_of_thrones,2200000.0,1.361E7,6447808.219178082
how_i_met_your_mother,6410000.0,1.313E7,9044182.692307692
lost,8289999.999999999,2.347E7,1.458099173553719E7
the_office,3250000.0,2.291E7,7358855.721393035
the_walking_dead,1190000.0,1.729E7,8137966.101694915


In [0]:
display(
    episodios
    .filter(
        (F.col("series_id") == "lost") &
        (F.col("season") == "6") &
        (F.col("episode_num_in_season") == "12")
    )
    .select(
        "series_id",
        "season",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "original_air_date",
        "prod_code",
        "us_viewers",
        "_source_file"
    )
)

series_id,season,episode_num_in_season,episode_num_overall,title,original_air_date,prod_code,us_viewers,_source_file
lost,6,12,105,LA X - Part 2,2010-02-02,null,12090000.0,dbfs:/Volumes/mvp_engdados/bronze/datafiles/lost_episodes.csv
lost,6,12,115,Everybody Loves Hugo,2010-04-13,null,9480000.0,dbfs:/Volumes/mvp_engdados/bronze/datafiles/lost_episodes.csv


In [0]:
display(
    imdb
    .filter(
        (F.col("series_id") == "lost") &
        (F.col("season") == "6") &
        (F.col("episode_num") == "12")
    )
    .select(
        "series_id",
        "season",
        "episode_num",
        "title",
        "original_air_date",
        "imdb_rating",
        "total_votes"
    )
)

series_id,season,episode_num,title,original_air_date,imdb_rating,total_votes
lost,6,12,Everybody Loves Hugo,13 Apr. 2010,8.4,4164


In [0]:
display(
    episodios
    .filter(
        (F.col("series_id") == "lost") &
        (F.col("season") == "6") &
        F.col("episode_num_overall").cast("int").between(103, 116)
    )
    .select(
        "season",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "original_air_date"
    )
    .orderBy(F.col("episode_num_overall").cast("int"))
)

season,episode_num_in_season,episode_num_overall,title,original_air_date
6,1,104,LA X - Part 1,2010-02-02
6,12,105,LA X - Part 2,2010-02-02
6,3,106,What Kate Does,2010-02-09
6,4,107,The Substitute,2010-02-16
6,5,108,Lighthouse,2010-02-23
6,6,109,Sundown,2010-03-02
6,7,110,Dr. Linus,2010-03-09
6,8,111,Recon,2010-03-16
6,9,112,Ab Aeterno,2010-03-23
6,10,113,The Package,2010-03-30


In [0]:
episodios_corrigidos = episodios.withColumn(
    "episode_num_in_season",
    F.when(
        (F.col("series_id") == "lost") &
        (F.col("season").cast("int") == 6) &
        (F.col("episode_num_overall").cast("int") == 105) &
        (F.col("title") == "LA X - Part 2"),
        F.lit("2")
    ).otherwise(F.col("episode_num_in_season"))
)

In [0]:
display(
    episodios_corrigidos
    .filter(
        (F.col("series_id") == "lost") &
        (F.col("season").cast("int") == 6) &
        (F.col("episode_num_overall").cast("int").between(104, 106))
    )
    .select(
        "season",
        "episode_num_in_season",
        "episode_num_overall",
        "title",
        "original_air_date"
    )
    .orderBy(F.col("episode_num_overall").cast("int"))
)

season,episode_num_in_season,episode_num_overall,title,original_air_date
6,1,104,LA X - Part 1,2010-02-02
6,2,105,LA X - Part 2,2010-02-02
6,3,106,What Kate Does,2010-02-09


In [0]:
display(
    episodios_corrigidos
    .groupBy("series_id", "season", "episode_num_in_season")
    .count()
    .filter(F.col("count") > 1)
    .orderBy("series_id", "season", "episode_num_in_season")
)

series_id,season,episode_num_in_season,count
